# Matrix Reasoning — Prompt Phase Experiments (InternVL3.5-2B)

This notebook documents a phased prompt engineering study on the **Matrix Reasoning** task
from the Levante benchmark.

**Task:** Each trial shows a 3×3 (or similar) matrix puzzle with one missing piece.
The model must identify the visual rule — changes in shape, size, shading, number, or
orientation across rows and columns — and select one of four options (A–D) that best
completes the pattern. Chance level: 25%.

**Model:** `OpenGVLab/InternVL3_5-2B-HF` — InternViT vision encoder + 2B InternLM language
model, images resized to 512px to avoid OOM on MPS.

**Motivation:** Most open-source VLMs score near 25–30% on Matrix Reasoning (essentially
chance), suggesting the task is visually demanding. InternVL3.5 uses a spatially-aware
vision encoder that may benefit from structured prompts that explicitly guide rule
extraction.


In [1]:
from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)

RESULTS = ROOT / "results" / "prompt_optimization" / "matrix-reasoning" / "internvl-3.5-2b"
_spec = importlib.util.spec_from_file_location(
    "experiment_matrix", ROOT / "scripts" / "prompt_optimization" / "matrix-reasoning" / "experiment_internvl2b.py"
)
mp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(mp)

manifest_rows = mp.load_manifest(ROOT / "data")
IMAGE_DIR = ROOT / "data" / "assets" / "2026-03-26" / "visual" / "matrix-reasoning"
TRIALS_LIST = mp.build_trials(manifest_rows, IMAGE_DIR)
TRIALS = {t["item_uid"]: t for t in TRIALS_LIST}
print(f"Loaded {len(TRIALS)} trials")


def build_prompt(phases: set[int], item_uid: str) -> str:
    t = TRIALS[item_uid]
    row = t["row"]
    p = mp.build_prompt_phase1(row, t["n_options"]) if 1 in phases else mp.build_prompt_baseline(row)
    if 4 in phases:
        p = mp.apply_phase4_rule_hint(p)
    if 5 in phases:
        p = mp.apply_phase5_rule_cot(p)
    return p


def load_result(csv_name: str, item_uid: str) -> dict | None:
    path = RESULTS / csv_name
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["item_uid"] == item_uid:
                return row
    return None


Loaded 79 trials


## Summary: All Phase Configurations

Accuracy, parse rate and delta vs baseline for all 9 configurations.


In [2]:
PHASE_CSVS = [
    ("phase_baseline.csv", "0 — Baseline"),
    ("phase_1.csv", "1 — Structured prompt"),
    ("phase_2.csv", "2 — Enhanced parsing"),
    ("phase_3.csv", "3 — Expert system prompt"),
    ("phase_4.csv", "4 — Rule enumeration hint"),
    ("phase_5.csv", "5 — Rule CoT (answer-last, 512 tok)"),
    ("phase_1_2_3.csv", "1+2+3 — Structural"),
    ("phase_1_2_3_4.csv", "1+2+3+4 — Structural + rule hint"),
    ("phase_1_2_3_5.csv", "1+2+3+5 — Structural + rule CoT"),
]


def summarize_all():
    rows = []
    baseline_acc = None
    for csv_name, label in PHASE_CSVS:
        path = RESULTS / csv_name
        if not path.exists():
            continue
        with open(path, newline="", encoding="utf-8") as f:
            data = list(csv.DictReader(f))
        n = len(data)
        if n == 0:
            continue
        correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
        parsed  = sum(1 for r in data if r["parsed"].lower() in ("true", "1"))
        acc = correct / n
        pr  = parsed / n
        if baseline_acc is None:
            baseline_acc = acc
        delta = "—" if baseline_acc is None or label.startswith("0") else f"{(acc - baseline_acc)*100:+.1f} pp"
        rows.append({"Phase": label, "N": n, "Accuracy": f"{acc:.1%}", "Parse %": f"{pr:.1%}", "Δ vs baseline": delta})
    return pd.DataFrame(rows)


df_summary = summarize_all()
print(df_summary.to_string(index=False))
df_summary.style.hide(axis="index")


                              Phase  N Accuracy Parse % Δ vs baseline
                       0 — Baseline 79    38.0%  100.0%             —
              1 — Structured prompt 79    40.5%  100.0%       +2.5 pp
               2 — Enhanced parsing 79    38.0%  100.0%       +0.0 pp
           3 — Expert system prompt 79    17.7%   68.4%      -20.3 pp
          4 — Rule enumeration hint 79    40.5%   89.9%       +2.5 pp
5 — Rule CoT (answer-last, 512 tok) 79    30.4%  100.0%       -7.6 pp
                 1+2+3 — Structural 79    35.4%  100.0%       -2.5 pp
   1+2+3+4 — Structural + rule hint 79    41.8%  100.0%       +3.8 pp
    1+2+3+5 — Structural + rule CoT 79    22.8%  100.0%      -15.2 pp


Phase,N,Accuracy,Parse %,Δ vs baseline
0 — Baseline,79,38.0%,100.0%,—
1 — Structured prompt,79,40.5%,100.0%,+2.5 pp
2 — Enhanced parsing,79,38.0%,100.0%,+0.0 pp
3 — Expert system prompt,79,17.7%,68.4%,-20.3 pp
4 — Rule enumeration hint,79,40.5%,89.9%,+2.5 pp
"5 — Rule CoT (answer-last, 512 tok)",79,30.4%,100.0%,-7.6 pp
1+2+3 — Structural,79,35.4%,100.0%,-2.5 pp
1+2+3+4 — Structural + rule hint,79,41.8%,100.0%,+3.8 pp
1+2+3+5 — Structural + rule CoT,79,22.8%,100.0%,-15.2 pp


---

## Phase 0 — Baseline

The manifest prompt: a single sentence asking to choose the missing puzzle piece,
followed by four option images. No explicit guidance on how to reason about the pattern.

**Prompt structure:**
```
Choose the best option to fill in the blank. Answer with A, B, C, or D.
A: <image1>; B: <image2>; C: <image3>; D: <image4>
[matrix image shown first]
```


In [3]:
uid0 = TRIALS_LIST[0]["item_uid"]
print("=== PROMPT (Baseline) ===")
print(build_prompt(set(), uid0))
print()
r = load_result("phase_baseline.csv", uid0)
if r:
    print(f"Correct: {r['correct_label']}  Predicted: {r['predicted_label']}  ✓={r['is_correct']}")
    print(f"Response: {r['raw_response'][:200]}")


=== PROMPT (Baseline) ===
<prompt_image> Choose the best option to fill in the blank. Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

Correct: C  Predicted: B  ✓=False
Response: B


---

## Phase 1 — Structured Prompt

Replaces the dense single-line manifest prompt with a clearer layout:
- Explicit instruction to look for the pattern
- Options listed one per line (A: / B: / C: / D:)
- No inline semicolons

This forces the model to treat each option as a distinct candidate.


In [4]:
uid1 = TRIALS_LIST[0]["item_uid"]
print("=== PROMPT (Phase 1) ===")
print(build_prompt({1}, uid1))
print()
r = load_result("phase_1.csv", uid1)
if r:
    print(f"Correct: {r['correct_label']}  Predicted: {r['predicted_label']}  ✓={r['is_correct']}")
    print(f"Response: {r['raw_response'][:200]}")


=== PROMPT (Phase 1) ===
<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.

Correct: C  Predicted: B  ✓=False
Response: B


---

## Phase 3 — Expert System Prompt

Adds a system prompt framing the model as a Raven's Progressive Matrices solver.
Explicitly mentions rule types (shape, size, shading, count, orientation) and
instructs the model to always respond with exactly one letter.


In [5]:
uid3 = TRIALS_LIST[2]["item_uid"]
print("=== SYSTEM PROMPT (Phase 3) ===")
print(mp.EXPERT_SYSTEM)
print()
print("=== USER PROMPT ===")
print(build_prompt({1, 3}, uid3))
print()
r = load_result("phase_3.csv", uid3)
if r:
    print(f"Correct: {r['correct_label']}  Predicted: {r['predicted_label']}  ✓={r['is_correct']}")
    print(f"Response: {r['raw_response'][:200]}")


=== SYSTEM PROMPT (Phase 3) ===
You are solving Raven's Progressive Matrices. Each puzzle shows a grid with a missing piece. Identify the visual rule — changes in shape, size, shading, number, or orientation — across rows and columns, then select the option that best completes the pattern. Always respond with exactly one letter: A, B, C, or D.

=== USER PROMPT ===
<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.

Correct: B  Predicted:   ✓=False
Response: To solve this puzzle, we need to identify the pattern in the arrangement of dots and their colors across the rows and columns.

1. **First Row**: 
   - Blue dots: 10
   - Yellow dots: 2
   - Black dot


---

## Phase 4 — Rule Enumeration Hint

Prepends an explicit hint asking the model to examine how properties change across
rows and columns before answering. This is a lightweight probe: does naming the
relevant dimensions (shape, size, shading, orientation) improve pattern detection?


In [6]:
uid4 = TRIALS_LIST[3]["item_uid"]
print("=== PROMPT (Phase 1+4) ===")
print(build_prompt({1, 4}, uid4))
print()
r = load_result("phase_4.csv", uid4)
if r:
    print(f"Correct: {r['correct_label']}  Predicted: {r['predicted_label']}  ✓={r['is_correct']}")
    print(f"Response: {r['raw_response'][:200]}")


=== PROMPT (Phase 1+4) ===
Hint: Examine how the shapes, sizes, shading, or orientation change across each row and each column. The same rule applies throughout the matrix.

<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.

Correct: D  Predicted: C  ✓=False
Response: C


---

## Phase 5 — Rule CoT with Answer-Last

Appends a 3-step chain-of-thought that forces the model to:
1. Identify the rule in Row 1
2. Verify it holds in Row 2
3. Apply it to Row 3 → missing piece

The reasoning ends with **"My answer is"** as the last line, so even if generation
is long, the answer letter is always at the end and easy to parse.

`max_new_tokens` is raised to 512 to allow the full reasoning chain to complete.


In [7]:
uid5 = TRIALS_LIST[4]["item_uid"]
print("=== PROMPT (Phase 1+5) ===")
print(build_prompt({1, 5}, uid5))
print()
r = load_result("phase_5.csv", uid5)
if r:
    print(f"Correct: {r['correct_label']}  Predicted: {r['predicted_label']}  ✓={r['is_correct']}")
    print(f"Response: {r['raw_response'][:200]}")


=== PROMPT (Phase 1+5) ===
<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.
Step 1: What visual property changes across Row 1 (shape, size, shading, count, orientation)?
Step 2: Does the same rule apply to Row 2?
Step 3: Apply the rule to Row 3 to identify the missing piece.
My answer is

Correct: C  Predicted: A  ✓=False
Response: Step 1: Analyze Row 1
- Shape: The objects are crescent-shaped.
- Size: The sizes vary, with some being larger and some smaller.
- Shading: The objects have different levels of shading.
- Count: There


---

## Combined Runs

### Phase 1+2+3 — Structural Improvements


In [8]:
uid_c1 = TRIALS_LIST[5]["item_uid"]
print("=== PROMPT (Phase 1+2+3) ===")
print(build_prompt({1, 2, 3}, uid_c1))
print()
r = load_result("phase_1_2_3.csv", uid_c1)
if r:
    print(f"Correct: {r['correct_label']}  Predicted: {r['predicted_label']}  ✓={r['is_correct']}")
    print(f"Response: {r['raw_response'][:200]}")


=== PROMPT (Phase 1+2+3) ===
<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.

Correct: A  Predicted: A  ✓=True
Response: A


### Phase 1+2+3+5 — Structural + Rule CoT

In [9]:
uid_c2 = TRIALS_LIST[6]["item_uid"]
print("=== PROMPT (Phase 1+2+3+5) ===")
print(build_prompt({1, 2, 3, 5}, uid_c2))
print()
r = load_result("phase_1_2_3_5.csv", uid_c2)
if r:
    print(f"Correct: {r['correct_label']}  Predicted: {r['predicted_label']}  ✓={r['is_correct']}")
    print(f"Response: {r['raw_response'][:200]}")


=== PROMPT (Phase 1+2+3+5) ===
<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.
Step 1: What visual property changes across Row 1 (shape, size, shading, count, orientation)?
Step 2: Does the same rule apply to Row 2?
Step 3: Apply the rule to Row 3 to identify the missing piece.
My answer is

Correct: C  Predicted: D  ✓=False
Response: <think>
Okay, let's see. I need to figure out the pattern in this matrix puzzle. The question is about Raven's Progressive Matrices, so the rule should be consistent across rows and columns.

First, l


---

## Conclusions — Matrix Reasoning (InternVL3.5-2B)

### Task description

Matrix Reasoning (Raven's Progressive Matrices) measures visual pattern completion. Each trial shows a 3x3 grid with a missing piece and four answer options. The model must identify the visual rule governing changes in shape, size, shading, count, or orientation across rows and columns, then select the option that best completes the pattern. There are 79 trials and 4 choices per trial (chance = 25%).

### Results summary (9 configurations)

| Phase | Accuracy | Parse % | Δ |
|-------|----------|---------|---|
| 0 — Baseline | 38.0% | 100% | — |
| 1 — Structured prompt | 40.5% | 100% | +2.5 pp |
| 2 — Enhanced parsing | 38.0% | 100% | +0.0 pp |
| 3 — Expert system prompt | 17.7% | 68.4% | −20.3 pp |
| 4 — Rule enumeration hint | 40.5% | 89.9% | +2.5 pp |
| 5 — Rule CoT (answer-last, 512 tok) | 30.4% | 100% | −7.6 pp |
| 1+2+3 — Structural | 35.4% | 100% | −2.5 pp |
| 1+2+3+4 — Structural + rule hint | 41.8% | 100% | +3.8 pp |
| 1+2+3+5 — Structural + rule CoT | 22.8% | 100% | −15.2 pp |

### Key findings

**1. Best configuration: 1+2+3+4 reaches 41.8% (+3.8 pp) — modest but consistent**
Combining structured prompt, enhanced parsing, expert system prompt, and rule enumeration hint yields the best result. The gain over baseline is small (+3.8 pp) and barely above chance (25%), suggesting prompt engineering has limited impact at this scale.

**2. Phase 3 (expert system prompt alone) is catastrophically bad (17.7%, parse rate 68.4%)**
The RPM-style system prompt ("identify the visual rule — shape, size, shading, number, orientation") causes InternVL3.5-2B to generate verbose CoT-like responses instead of a single letter. The model fails to parse its own output 31.6% of the time. When used inside 1+2+3 (which adds enhanced parsing), the parse failure disappears but accuracy still drops (35.4% vs 38.0% baseline), showing the instruction itself confuses the model.

**3. Rule CoT (Phase 5) is consistently harmful**
Phase 5 alone drops to 30.4% (−7.6 pp). Combined with the structural base (1+2+3+5), it falls to 22.8% (−15.2 pp) — barely above chance. Forcing step-by-step spatial reasoning in text does not help InternVL3.5-2B solve visual pattern recognition; it appears to lead the model away from the visual evidence.

**4. The rule enumeration hint (Phase 4) is the most reliable single addition (+2.5 pp)**
Prepending a one-line hint ("Examine how shapes, sizes, shading, or orientation change across each row and column") reliably adds +2.5 pp without hurting parse rate. It works both alone and combined with the structural base.

**5. Prompt engineering reaches a hard ceiling at ~42%**
The best result (41.8%) is only 16.8 pp above chance (25%). All structural improvements plateau within a few percentage points of each other. This strongly suggests the bottleneck is visual feature extraction — at 2B scale, InternVL3.5 cannot reliably perceive the subtle geometric rules in RPM puzzles regardless of how the question is framed.

### Production recommendation

Given the limited gains, Phase 1+2+3+4 (41.8%) is the recommended configuration if prompt optimization is used at all:
- Structured prompt separating the matrix and options
- Rule enumeration hint as a one-line preamble
- Expert system prompt via system channel
- `max_new_tokens = 64`

However, the ~4 pp gain is unlikely to be meaningful in practice. A larger model (8B+) or a model with a stronger spatial visual encoder would likely be required for meaningful progress on this task.

### Limitations

- Only InternVL3.5-2B was tested; a larger model (8B) may handle RPM-style reasoning better
- 79 trials — differences of ≤3 items (≤4 pp) may reflect noise
- No few-shot examples were tested; providing 1-2 solved examples might help more than text-based hints
- Image resize to 512px may discard fine-grained details relevant for some patterns
